# Phase 2 · Train with the reward (Kaggle, 2× T4)

**This notebook is a launcher. You don't need to edit it.** It pulls the latest code and runs `scripts/p2_run.py`, which follows the plan in [`configs/p2/run.yaml`](https://github.com/guohere/machine-translation-with-RLHF/blob/claude/compassionate-ritchie-2nga86/configs/p2/run.yaml). The plan lives in the repository, so changes arrive with `git pull`, with no editing or re-importing.

**How to run:** click **Save Version → Save & Run All (Commit)** and close the laptop. Each run does the next block of work and ends with "Next run: …"; when it finishes, run it again the same way. Every step skips what is already saved: adapters in your private Hugging Face repo, results in `results/p2/` on GitHub. If a step fails, the run stops and pushes the full log to `results/p2/<model>/logs/`.

What the runs do, in order:
1. **Qwen3-1.7B, Part A:** SFT, the sampling pilot (one temperature / top-p for every sampling-based method), 5,000 × 8 samples scored with MetricX-QE, RSFT and DPO, translations, including OPUS-MT and best-of-8.
2. **Qwen3-1.7B, Part B:** GRPO and guarded GRPO side by side on the two GPUs, translations, checkpoint curves.
3. **Gemma 3 1B, Part A:** with a smoke test first, as it's a new model.
4. **Gemma 3 1B, Part B.**

### Setup (once)
- Kaggle *Settings*: Accelerator **GPU T4 ×2**, Internet **On**.
- *Add-ons → Secrets*:
  - **`HF_TOKEN`**: a Hugging Face *write* token whose account has accepted the Gemma 3 and CometKiwi licenses.
  - **`GH_TOKEN`**: a fine-grained GitHub token for this repository with *Contents: Read and write*.

  Attach both to this notebook.

In [ ]:
# ---- configuration: normally nothing to change ----
REPO = "https://github.com/guohere/machine-translation-with-RLHF.git"
BRANCH = "claude/compassionate-ritchie-2nga86"
OVERRIDES = ""   # optional, overrides configs/p2/run.yaml, e.g. "part=B" or "model=gemma3-1b smoke_test=true"

In [ ]:
# ---- GPU check and Hugging Face token (works on Kaggle and Colab) ----
import os, subprocess

WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.path.expanduser("~"))
REPO_DIR = os.path.join(WORK, "machine-translation-with-RLHF")   # every later cell runs from here
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the accelerator to a T4 GPU before continuing")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded")
else:
    print("WARNING: no HF_TOKEN secret found - the gated CometKiwi model will fail to download")
gh = get_secret("GH_TOKEN")
if gh:
    os.environ["GH_TOKEN"] = gh   # the driver pushes results to GitHub after every step
    print("GH_TOKEN loaded")
else:
    print("WARNING: no GH_TOKEN secret - results stay in this session only")

In [ ]:
# ---- get the code and install (~3 minutes) ----
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
os.chdir(REPO_DIR)
!git log -1 --format="code version: %h %s"
# transformers stays below 5 so the COMET evaluator works next to TRL (see pyproject.toml)
!pip install -q -e ".[phase1,phase2]"
!pip install -q --no-deps unbabel-comet==2.2.7 "git+https://github.com/google-research/mt-metrics-eval"
!python -c "import torch, transformers, trl, peft, comet; print('ready: torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| GPUs', torch.cuda.device_count())"

### Run the plan
The output streams below. In a background run (Save & Run All), it's under the version's **Logs** tab.

In [ ]:
import subprocess
os.chdir(REPO_DIR)
cmd = f"python scripts/p2_run.py --branch {BRANCH} " + (f"--set {OVERRIDES}" if OVERRIDES.strip() else "")
print("$", cmd, flush=True)
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("The run stopped at a failed step; its log was pushed to results/p2/<model>/logs/ on GitHub.")

In [ ]:
# ---- quick evaluation of the model this run worked on ----
import glob
from IPython.display import Markdown, display
reports = sorted(glob.glob("results/p2/*/outputs/quick_eval.md"), key=os.path.getmtime)
if reports:
    print(reports[-1])
    display(Markdown(open(reports[-1]).read()))

### Troubleshooting
| Symptom | Fix |
|---|---|
| `Permission 'kernelSessions.enableInternet' was denied` | Verify your phone number in Kaggle (profile picture → *Settings*), then *Settings → Internet → On* |
| `Hub mirror disabled` | `HF_TOKEN` is read-only: create a *write* token |
| `401` / gated repo (`gemma-3-1b-it`, `wmt22-cometkiwi-da`) | Accept the model's license on Hugging Face with the account of `HF_TOKEN` |
| `push failed` | Check `GH_TOKEN` (fine-grained, this repository, *Contents: Read and write*) |
| Anything else | The failed step's log is in `results/p2/<model>/logs/` on GitHub; ask Claude to read it |